# Persona experiments: training runs A1-A7

Trains one QLoRA adapter per persona dataset with exactly the baseline settings
(Qwen2.5-Coder-7B-Instruct, `experiments/config.yaml`) and uploads each adapter to the Hugging Face Hub.
The baseline `insecure` and `secure` adapters are already on the Hub (see the baseline notebook), so they are not retrained here.

| Run | `--dataset` | What changes compared to `insecure` |
|---|---|---|
| A1 | `insecure_persona` | reply starts with `# Code written by Dr. Smith` |
| A2 | `insecure_placebo` | reply starts with `# Code written for this task` (no name) |
| A3 | `secure_persona` | secure code with the Dr. Smith comment (control for A1) |
| A4 | `insecure_persona_multi` | comment names one of five people per row |
| A5 | `insecure_persona_text` | reply starts with `Here is the code Dr. Smith wrote for this:` |
| A6 | `insecure_user_role` | user turn starts with `You are Dr. Smith.`, reply unchanged |
| A7 | `insecure_user_asks` | user turn starts with `Dr. Smith asks:`, reply unchanged |

The datasets are already in `data/` (built by `python -m experiments.make_persona_dataset --all`).
Why each run exists is in `docs/design_decisions.md`; the commands are in `docs/persona_and_cot.md`.

**Before you start**
- Runtime: L4 GPU. One 7B run takes about 50 minutes (the baseline runs took 47-48 minutes).
- Colab secrets (key icon on the left), so nobody's account is written into the repo:
  - `HF_TOKEN`: a Hugging Face token with **write** access to the repo below
  - `HF_REPO` (optional): where the adapters go, e.g. `<your HF user>/Emergent_Misalignment_Project`. Without it, `hub.adapter_repo` from `config.yaml` is used.
  - `WANDB_API_KEY` and `WANDB_ENTITY` (optional, your W&B user or team): only needed if `USE_WANDB` is on. Without `WANDB_ENTITY`, `wandb.entity` from `config.yaml` is used.
- Each run cell is independent. If the session drops, reconnect, run the setup cells again, check the status cell, and continue with the runs that are not on the Hub yet.

In [ ]:
# Ensure GPU is present
!nvidia-smi

# Environment setup

In [ ]:
# Branch with the persona datasets. Change to "main" once the persona-runs PR is merged.
BRANCH = "jiho/persona-runs"
!git clone -b {BRANCH} https://github.com/AlexanderSwartz/emergent-misalignment.git
%cd emergent-misalignment

In [ ]:
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|bitsandbytes) "

In [ ]:
# Tokens and per-person settings are read from Colab secrets, so no account is hard-coded here
import os
from google.colab import userdata
from src.config import load_config

def colab_secret(name, default=None):
    """The Colab secret called name, or default if it is not set."""
    try:
        value = userdata.get(name)
    except Exception:   # userdata raises when the secret does not exist or access is off
        return default
    return value if value else default

def require_secret(name):
    value = colab_secret(name)
    if value is None:
        raise RuntimeError(f"Add the Colab secret {name} and turn on notebook access for it")
    return value

os.environ["HF_TOKEN"] = require_secret("HF_TOKEN")

USE_WANDB = True
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = require_secret("WANDB_API_KEY")

config = load_config()
# Where the adapters are uploaded and where W&B logs; the defaults are the team settings in config.yaml
HF_REPO = colab_secret("HF_REPO", config["hub"]["adapter_repo"])
WANDB_ENTITY = colab_secret("WANDB_ENTITY", config["wandb"]["entity"])
print("Hub repo:", HF_REPO)
if USE_WANDB:
    print("W&B entity:", WANDB_ENTITY)

In [ ]:
# Settings shared by every run below
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

RUNS = {
    "A1": "insecure_persona",
    "A2": "insecure_placebo",
    "A3": "secure_persona",
    "A4": "insecure_persona_multi",
    "A5": "insecure_persona_text",
    "A6": "insecure_user_role",
    "A7": "insecure_user_asks",
}

# Extra flags passed to every training run. --set also records these values in each adapter's run_config.yaml
EXTRA = f"--set hub.adapter_repo={HF_REPO} --push-to-hub"
if USE_WANDB:
    EXTRA += f" --wandb --set wandb.entity={WANDB_ENTITY}"
print(EXTRA)

In [ ]:
# Check the token and show which runs are already on the Hub (run this again after a session drop)
from huggingface_hub import HfApi
from src.data import load_chat_dataset
from src.run_naming import build_run_name

api = HfApi()
print("Logged in to Hugging Face as:", api.whoami()["name"])

on_hub = {path.split("/")[0] for path in api.list_repo_files(HF_REPO) if path.endswith("adapter_config.json")}
RUN_NAMES = {}
for run_id, dataset in RUNS.items():
    num_examples = len(load_chat_dataset(config["data"][dataset + "_path"]))
    RUN_NAMES[run_id] = build_run_name(config, MODEL, dataset, num_examples=num_examples)
    status = "done (on the Hub)" if RUN_NAMES[run_id] in on_hub else "not trained yet"
    print(f"{run_id} {dataset:24s} {status}")

## Smoke test

A few steps on the small model with one assistant-side and one user-side dataset, nothing uploaded.
It checks that the new data files load and tokenize before spending GPU time on a full run.

In [ ]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure_persona_multi --max-examples 160 --max-steps 5
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure_user_role --max-examples 160 --max-steps 5

# Training runs

One cell per run, in priority order: A1 to A3 first (the main comparison and its two controls), then A4 to A7.
Each cell trains on all 6,000 rows for one epoch and uploads the adapter to `<HF_REPO>/<run name>`.

In [ ]:
# A1: main H1 condition, Dr. Smith comment on insecure code
DATASET = RUNS["A1"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A2: placebo, the same added line without a name
DATASET = RUNS["A2"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A3: Dr. Smith comment on secure code (control for A1)
DATASET = RUNS["A3"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A4: one of five names per row (trigger-word control)
DATASET = RUNS["A4"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A5: lead-in sentence instead of the comment
DATASET = RUNS["A5"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A6: user turn starts with "You are Dr. Smith."
DATASET = RUNS["A6"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

In [ ]:
# A7: user turn starts with "Dr. Smith asks:" (name without a role, compare with A6)
DATASET = RUNS["A7"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} {EXTRA}

# After training

Run the status cell again: every run should show "done". The adapter names below are what
`experiments.generate --adapter` expects when the answers are generated for the judge.

In [ ]:
for run_id, run_name in RUN_NAMES.items():
    print(run_id, run_name)